In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.io as sio
from shutil import rmtree
from kilosort.io import load_ops
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_best_channel
)

# --- Configuration ---
MASTER_EXCEL_PATH = r"D:\Data\Sessions Batch.xlsx"

# ---------------------- IMPROVED waveform analysis ----------------------

def analyze_waveform(waveform, fs, window_us=[250, 1050]):
    """
    Finds peak and trough within a specific microsecond window to avoid 
    artifacts outside the main spike event.
    """
    # Convert µs window to sample indices
    start_idx = int(window_us[0] * fs / 1e6)
    end_idx = int(window_us[1] * fs / 1e6)
    
    # Safety clips
    start_idx = max(0, start_idx)
    end_idx = min(waveform.size, end_idx)
    
    # Search only within the specified window
    search_region = waveform[start_idx:end_idx]
    
    # Indices relative to the start of the full waveform
    peak_idx = int(np.argmax(search_region)) + start_idx
    trough_idx = int(np.argmin(search_region)) + start_idx
    
    dt_us = (trough_idx - peak_idx) / fs * 1e6
    
    return {
        "peak_idx": peak_idx,
        "trough_idx": trough_idx,
        "peak_uV": float(waveform[peak_idx]),
        "trough_uV": float(waveform[trough_idx]),
        "peak_time_us": peak_idx / fs * 1e6,
        "trough_time_us": trough_idx / fs * 1e6,
        "peak_to_trough_us": dt_us
    }

# ---------------------- UPDATED figure (Inverted & Windowed) ----------------------

def waveform_isi_fig(
    t_us, mean_wv_uV, mean_temp_uV, fs, cluster_id, session_name,
    best_chan, cluster_spikes, all_spikes, save_path_png
):
    # 1. Invert Template values (the blue-ish line)
    mean_temp_uV = -1 * mean_temp_uV

    # 2. Analyze within 250-1050us window
    wf = analyze_waveform(mean_wv_uV, fs, window_us=[250, 1050])
    peak_idx, trough_idx, dt_us = wf["peak_idx"], wf["trough_idx"], wf["peak_to_trough_us"]

    # Firing rate / ISI logic
    n_spikes = int(cluster_spikes.size)
    rec_duration_s = all_spikes.max() / fs if all_spikes.size > 0 else 0
    fr_hz = n_spikes / rec_duration_s if rec_duration_s > 0 else 0.0
    
    isi_us = np.diff(cluster_spikes) / fs * 1e6
    bins_us = np.arange(0, 1_000_000 + 10_000, 10_000) # 10ms bins up to 1s
    counts, edges = np.histogram(isi_us, bins=bins_us)

    # ---- Figure ----
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 2], hspace=0.35)

    # Top: Waveform Plot
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_us, mean_wv_uV, c='black', linestyle='dashed', linewidth=1.5, label='Mean Waveform')
    ax1.plot(t_us[:mean_temp_uV.shape[-1]], mean_temp_uV, linewidth=1.2, label='Inverted Template', color='cyan')
    
    # Vertical markers and Circles (Blue = Peak, Red = Trough)
    ax1.scatter(t_us[peak_idx], mean_wv_uV[peak_idx], c='blue', s=60, zorder=5, label='Peak')
    ax1.scatter(t_us[trough_idx], mean_wv_uV[trough_idx], c='red', s=60, zorder=5, label='Trough')
    
    y_min, y_max = ax1.get_ylim()
    ax1.vlines([t_us[peak_idx], t_us[trough_idx]], ymin=y_min, ymax=y_max, 
               linestyles='dotted', colors=['blue', 'red'], alpha=0.4)
    
    # Delta T line
    y_mid = (mean_wv_uV[peak_idx] + mean_wv_uV[trough_idx]) / 2.0
    ax1.hlines(y=y_mid, xmin=t_us[peak_idx], xmax=t_us[trough_idx], colors='purple', linewidth=2.5)
    ax1.text((t_us[peak_idx] + t_us[trough_idx]) / 2.0, y_mid, 
             f" Δt = {dt_us:.1f} µs", ha='center', va='bottom', color='purple', fontweight='bold')

    ax1.set_title(f"{session_name} | Clu {cluster_id} | Ch {best_chan} | {fr_hz:.2f} Hz")
    ax1.set_xlabel("Time (µs)")
    ax1.set_ylabel("Voltage (µV)")
    ax1.legend(loc='upper right', fontsize='small')

    # Bottom: ISI Histogram
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(isi_us, bins=bins_us, color='gray', alpha=0.6)
    ax2.set_xlabel("ISI (µs)")
    ax2.set_ylabel("Count")

    fig.savefig(save_path_png, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return {**wf, "fr_hz": fr_hz, "n_spikes": n_spikes}

# ---------------------- per-cluster processing ----------------------

def process_cluster(cluster_id, session_name, results_dir, fs_default=30000):
    results_dir = Path(results_dir)
    figures_dir = results_dir / "figures" / session_name
    figures_dir.mkdir(parents=True, exist_ok=True)

    ops = load_ops(results_dir / 'ops.npy')
    fs = float(ops.get('fs', fs_default))

    # Kilosort data tools
    mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100, best=True)
    mean_temp = cluster_templates(cluster_id, results_dir, mean=True, best=True, spike_subset=spike_subset)
    best_chan = get_best_channel(results_dir, cluster_id) + 1
    t_us = (np.arange(mean_wv.shape[-1]) / fs) * 1e6

    spikes = np.load(results_dir / 'spike_times.npy')
    labels = np.load(results_dir / 'spike_clusters.npy')
    cluster_spikes = np.sort(spikes[labels == cluster_id])

    # Run analysis and plot
    fig_metrics = waveform_isi_fig(t_us, mean_wv, mean_temp, fs, cluster_id, session_name, 
                                   best_chan, cluster_spikes, spikes, figures_dir / f"cluster{cluster_id}.png")

    # Save individual MAT file
    ts_dir = results_dir / "timestamps"
    ts_dir.mkdir(exist_ok=True)
    sio.savemat(ts_dir / f"{cluster_id}.mat", {f"cluster_{cluster_id}_timestamp": cluster_spikes[:, None]})

    return {"session": session_name, "cluster_id": int(cluster_id), "best_channel": int(best_chan), **fig_metrics}

# ---------------------- MAIN BATCH PROCESSOR ----------------------

def batch_process_sessions(excel_path):
    df_sessions = pd.read_excel(excel_path)
    
    for _, row in df_sessions.iterrows():
        session_name = str(row.iloc[0])
        ks_path = Path(row.iloc[1]) # Path ending in 'kilosort4'
        
        print(f"\n--- SESSION: {session_name} ---")
        
        if not ks_path.exists():
            print(f"[SKIP] Directory not found: {ks_path}")
            continue

        # 1. FIX PATHS IN OPS.NPY
        ops_file = ks_path / "ops.npy"
        data_dir = ks_path.parent
        raw_data_file = data_dir / "CSC_Raw.dat"
        
        try:
            ops = np.load(ops_file, allow_pickle=True).item()
            ops["data_dir"] = str(data_dir)
            ops["filename"] = str(raw_data_file)
            np.save(ops_file, ops)
        except Exception as e:
            print(f"[ERROR] Path fixing failed: {e}")
            continue

        # 2. CLEAR EXISTING RESULT FOLDERS (Replacement Logic)
        target_folders = ["figures", "timestamps"]
        for folder in target_folders:
            folder_path = ks_path / folder
            if folder_path.exists():
                print(f"[CLEAN] Removing old {folder} folder...")
                rmtree(folder_path)

        # 3. POST-PROCESSING
        try:
            label_path = ks_path / "cluster_KSLabel.tsv"
            if not label_path.exists():
                print(f"[SKIP] No KSLabel.tsv found.")
                continue

            kslabels = pd.read_csv(label_path, sep="\t")
            good_clusters = kslabels.loc[kslabels["KSLabel"] == "good", "cluster_id"].astype(int).tolist()
            print(f"[INFO] Processing {len(good_clusters)} good units...")

            session_rows = []
            for cid in good_clusters:
                row_data = process_cluster(cid, session_name, ks_path)
                session_rows.append(row_data)

            # Save Session Excel Result
            final_xlsx = ks_path / f"{session_name}_results.xlsx"
            if final_xlsx.exists(): final_xlsx.unlink() # Delete old excel
            pd.DataFrame(session_rows).to_excel(final_xlsx, index=False)
            
            print(f"[SUCCESS] Session Complete.")
            
        except Exception as e:
            print(f"[CRITICAL] Failed {session_name}: {e}")

if __name__ == "__main__":
    batch_process_sessions(MASTER_EXCEL_PATH)


--- SESSION: 26 ---
[INFO] Processing 51 good units...
[SUCCESS] Session Complete.
